In [1]:
import json, csv, time, zipfile
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent
EXTRACT_DIR = PROJECT_DIR / "data" / "sen2fire"

In [2]:
REF_ZIP = PROJECT_DIR / "data" / "reference" / "Baseline_code.zip"
REF_ZIP.parent.mkdir(parents=True, exist_ok=True)
print("Folder ready :", REF_ZIP.parent)
print("Zip present  :", REF_ZIP.exists())

if REF_ZIP.exists():
    with zipfile.ZipFile(REF_ZIP) as zf:
        for info in zf.infolist():
            print(f"{info.file_size:>10,d}  {info.filename}")
    with zipfile.ZipFile(REF_ZIP) as zf:
        text = zf.read("dataset/Sen2Fire_Dataset.py").decode("utf-8", errors="replace")

        for n, line in enumerate(text.splitlines(), start=1):
            print(f"{n:3d}  {line}")

Folder ready : g:\sen2fire-thesis\data\reference
Zip present  : True
         0  dataset/__init__.py
     7,083  dataset/Sen2Fire_Dataset.py
    15,721  dataset/test.txt
    45,682  dataset/train.txt
    15,676  dataset/val.txt
         0  model/__init__.py
     3,157  model/Networks.py
     8,301  Test.py
    13,787  Train.py
         0  utils/__init__.py
     2,769  utils/tools.py
  1  import os.path as osp
  2  import numpy as np
  3  from torch.utils import data
  4  from tqdm import tqdm
  5  
  6  modename = ['all_bands',                        #0
  7              'all_bands_aerosol',                #1
  8              'rgb',                              #2
  9              'rgb_aerosol',                      #3
 10              'swir',                             #4
 11              'swir_aerosol',                     #5
 12              'nbr',                              #6
 13              'nbr_aerosol',                      #7   
 14              'ndvi',                     

In [3]:
splits = {}
with zipfile.ZipFile(REF_ZIP) as zf:
    for name in ["train", "val", "test"]:
        raw = zf.read(f"dataset/{name}.txt").decode("utf-8")
        splits[name] = [line.strip() for line in raw.splitlines() if line.strip()]

for name, items in splits.items():
    scenes = sorted({Path(i).parts[0] if len(Path(i).parts) > 1 else "(no folder)" for i in items})
    print(f"{name:5s}: {len(items):5d} entries | first: {items[0]} | folders: {scenes}")

all_on_disk = {f"{s}/{p.name}" for s in ["scene1", "scene2", "scene3", "scene4"]
               for p in (EXTRACT_DIR / s).glob("*.npz")}

for name, items in splits.items():
    norm = [i.replace("\\", "/") for i in items]
    missing = [i for i in norm if i not in all_on_disk]
    print(f"{name:5s}: missing on disk = {len(missing)}", missing[:3])

sets = {k: {i.replace("\\", "/") for i in v} for k, v in splits.items()}
print("train & val overlap :", len(sets["train"] & sets["val"]))
print("train & test overlap:", len(sets["train"] & sets["test"]))
print("val & test overlap  :", len(sets["val"] & sets["test"]))
print("on disk but in no split:", len(all_on_disk - sets["train"] - sets["val"] - sets["test"]))

train:  1458 entries | first: scene1/scene_1_patch_1_1.npz | folders: ['scene1', 'scene2']
val  :   504 entries | first: scene3/scene_3_patch_1_1.npz | folders: ['scene3']
test :   504 entries | first: scene4/scene_4_patch_1_1.npz | folders: ['scene4']
train: missing on disk = 0 []
val  : missing on disk = 0 []
test : missing on disk = 0 []
train & val overlap : 0
train & test overlap: 0
val & test overlap  : 0
on disk but in no split: 0


In [4]:
import time

W_B12, N_B12 = 0.02, 50      # B12 bins: 0.00-1.00 in steps of 0.02
W_R,   N_R   = 0.05, 60      # ratio bins: 0.00-3.00 in steps of 0.05
HIST_PATH = PROJECT_DIR / "data" / "baseline_hist.npz"

def build_hist(items, label=""):
    H = np.zeros((len(items), 2, N_B12 * N_R), dtype=np.int32)
    pos = np.zeros(len(items), dtype=np.int64)
    t0 = time.perf_counter()
    for i, rel in enumerate(items):
        with np.load(EXTRACT_DIR / rel) as d:
            img = d["image"]
            lb = d["label"] > 0
        b11 = img[10].astype(np.float32) / 10000.0
        b12 = img[11].astype(np.float32) / 10000.0
        valid = b11 > 0.05
        ratio = b12 / np.maximum(b11, 1e-6)
        flat = (np.clip((b12 / W_B12).astype(np.int32), 0, N_B12 - 1) * N_R
                + np.clip((ratio / W_R).astype(np.int32), 0, N_R - 1))
        H[i, 1] = np.bincount(flat[valid & lb],  minlength=N_B12 * N_R)
        H[i, 0] = np.bincount(flat[valid & ~lb], minlength=N_B12 * N_R)
        pos[i] = lb.sum()
        if (i + 1) % 300 == 0:
            print(f"{label}: {i + 1}/{len(items)}  ({time.perf_counter() - t0:.0f} s)")
    return H, pos

if HIST_PATH.exists():
    print("Cache already exists, skipping.")
else:
    Htr, ptr = build_hist(splits["train"], "train")
    Hva, pva = build_hist(splits["val"], "val")
    np.savez_compressed(HIST_PATH, Htr=Htr, ptr=ptr, Hva=Hva, pva=pva)
    print("Saved", HIST_PATH)
    print("Label pixels  train:", int(ptr.sum()), "| val:", int(pva.sum()))

Cache already exists, skipping.


In [5]:
d = np.load(HIST_PATH)
Htr, ptr = d["Htr"], d["ptr"]
pos_total = float(ptr.sum())

pos_hist = Htr[:, 1].sum(axis=0).reshape(N_B12, N_R).astype(np.float64)
neg_hist = Htr[:, 0].sum(axis=0).reshape(N_B12, N_R).astype(np.float64)

def tail_sum(h):
    return h[::-1, ::-1].cumsum(axis=0).cumsum(axis=1)[::-1, ::-1]

TP = tail_sum(pos_hist)
FP = tail_sum(neg_hist)
FN = pos_total - TP
prec = TP / np.maximum(TP + FP, 1)
rec  = TP / pos_total
f1   = 2 * TP / np.maximum(2 * TP + FP + FN, 1)
iou  = TP / np.maximum(TP + FP + FN, 1)

def show(a, r, tag=""):
    print(f"{tag:22s} B12>={a*W_B12:.2f}  ratio>={r*W_R:.2f} | "
          f"precision={prec[a, r]:.3f}  recall={rec[a, r]:.3f}  F1={f1[a, r]:.3f}  IoU={iou[a, r]:.3f}")

all_px = float(pos_hist.sum() + neg_hist.sum())
p0 = pos_total / all_px
print(f"Fire share of training pixels: {p0:.4f}  -> 'flag everything' F1 = {2*p0/(1+p0):.3f}\n")

a, r = np.unravel_index(np.argmax(f1), f1.shape)
show(a, r, "best overall (F1)")
a0 = np.argmax(f1[:, 0])
show(a0, 0, "best brightness only")

print("\nTop 8 combinations by F1:")
for idx in np.argsort(f1.ravel())[::-1][:8]:
    a, r = np.unravel_index(idx, f1.shape)
    show(a, r)

Fire share of training pixels: 0.0250  -> 'flag everything' F1 = 0.049

best overall (F1)      B12>=0.08  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
best brightness only   B12>=0.04  ratio>=0.00 | precision=0.025  recall=0.996  F1=0.049  IoU=0.025

Top 8 combinations by F1:
                       B12>=0.08  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.00  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.06  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.04  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.02  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.10  ratio>=1.15 | precision=0.195  recall=0.033  F1=0.056  IoU=0.029
                       B12>=0.12  ratio>=1.15 | precision=0.196  recall=0.033  F1=0.056  IoU=0.029
          

In [6]:
n = Htr.shape[0]
tot = (Htr[:, 0] + Htr[:, 1]).reshape(n, N_B12, N_R).astype(np.int64)
tail = tot[:, ::-1, ::-1].cumsum(axis=1).cumsum(axis=2)[:, ::-1, ::-1]
y = ptr > 0
p_fire = y.mean()
print(f"Train patches: {n} | with fire label: {y.sum()} ({p_fire:.3f}) | "
      f"'flag all' F1 = {2*p_fire/(1+p_fire):.3f}\n")

best = None
for k in [1, 2, 5, 10, 20, 50, 100, 200]:
    pred = tail >= k
    TPp = (pred & y[:, None, None]).sum(axis=0)
    FPp = (pred & ~y[:, None, None]).sum(axis=0)
    FNp = y.sum() - TPp
    f1p = 2 * TPp / np.maximum(2 * TPp + FPp + FNp, 1)
    a, r = np.unravel_index(np.argmax(f1p), f1p.shape)
    pr = TPp[a, r] / max(TPp[a, r] + FPp[a, r], 1)
    rc = TPp[a, r] / y.sum()
    print(f"k={k:<4d} B12>={a*W_B12:.2f}  ratio>={r*W_R:.2f} | "
          f"precision={pr:.3f}  recall={rc:.3f}  F1={f1p[a, r]:.3f}")
    if best is None or f1p[a, r] > best[0]:
        best = (f1p[a, r], k, a, r)

print(f"\nBest patch-level rule: k={best[1]}, B12>={best[2]*W_B12:.2f}, ratio>={best[3]*W_R:.2f}, F1={best[0]:.3f}")

Train patches: 1458 | with fire label: 153 (0.105) | 'flag all' F1 = 0.190

k=1    B12>=0.94  ratio>=1.30 | precision=0.714  recall=0.490  F1=0.581
k=2    B12>=0.94  ratio>=1.30 | precision=0.714  recall=0.490  F1=0.581
k=5    B12>=0.00  ratio>=2.25 | precision=0.659  recall=0.529  F1=0.587
k=10   B12>=0.00  ratio>=2.20 | precision=0.687  recall=0.516  F1=0.590
k=20   B12>=0.68  ratio>=2.60 | precision=0.807  recall=0.464  F1=0.589
k=50   B12>=0.00  ratio>=1.60 | precision=0.738  recall=0.497  F1=0.594
k=100  B12>=0.32  ratio>=1.30 | precision=0.818  recall=0.471  F1=0.598
k=200  B12>=0.24  ratio>=1.45 | precision=0.886  recall=0.458  F1=0.603

Best patch-level rule: k=200, B12>=0.24, ratio>=1.45, F1=0.603


In [7]:
import json

def patch_tail(H):
    m = H.shape[0]
    t = (H[:, 0] + H[:, 1]).reshape(m, N_B12, N_R).astype(np.int64)
    return t[:, ::-1, ::-1].cumsum(axis=1).cumsum(axis=2)[:, ::-1, ::-1]

def patch_f1_grid(tail, y, k):
    pred = tail >= k
    TPp = (pred & y[:, None, None]).sum(axis=0)
    FPp = (pred & ~y[:, None, None]).sum(axis=0)
    FNp = y.sum() - TPp
    return 2 * TPp / np.maximum(2 * TPp + FPp + FNp, 1)

ks = [1, 2, 5, 10, 20, 50, 100, 200, 300, 500, 1000, 2000, 5000]
cands = []
for k in ks:
    g = patch_f1_grid(tail, y, k)
    a, r = np.unravel_index(np.argmax(g), g.shape)
    cands.append((float(g[a, r]), k, int(a), int(r)))
    if k >= 200:
        print(f"train  k={k:<5d} B12>={a*W_B12:.2f} ratio>={r*W_R:.2f}  F1={g[a, r]:.3f}")

f1_best, k_b, a_b, r_b = max(cands)
rule = dict(level="patch", k=int(k_b), b12_min=round(a_b * W_B12, 2), ratio_min=round(r_b * W_R, 2),
            train_f1=round(f1_best, 4), tuned_on="train scenes only; F1 of fire class")
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "results" / "frozen_rule_v1.json").write_text(json.dumps(rule, indent=2))
print("\nFROZEN:", rule)

Hva, pva = d["Hva"], d["pva"]
tailv = patch_tail(Hva)
yv = pva > 0
pred = tailv[:, a_b, r_b] >= k_b
TPv = int((pred & yv).sum()); FPv = int((pred & ~yv).sum())
FNv = int((~pred & yv).sum()); TNv = int((~pred & ~yv).sum())
pv, rv = TPv / max(TPv + FPv, 1), TPv / max(TPv + FNv, 1)
pf = yv.mean()
print(f"\nVAL patch-level: n={len(yv)}, fire patches={int(yv.sum())} ({pf:.3f}), 'flag all' F1={2*pf/(1+pf):.3f}")
print(f"TP={TPv} FP={FPv} FN={FNv} TN={TNv} | precision={pv:.3f} recall={rv:.3f} F1={2*pv*rv/max(pv+rv,1e-9):.3f}")

ap, rp = 5, 23
def pix_metrics(Hs, ps):
    pos_h = Hs[:, 1].sum(axis=0).reshape(N_B12, N_R).astype(np.float64)
    neg_h = Hs[:, 0].sum(axis=0).reshape(N_B12, N_R).astype(np.float64)
    TPx, FPx = pos_h[ap:, rp:].sum(), neg_h[ap:, rp:].sum()
    FNx = ps.sum() - TPx
    return TPx / (TPx + FPx), TPx / ps.sum(), 2 * TPx / (2 * TPx + FPx + FNx), TPx / (TPx + FPx + FNx)
print("VAL pixel-level (B12>=0.10, ratio>=1.15): precision=%.3f recall=%.3f F1=%.3f IoU=%.3f" % pix_metrics(Hva, pva))

train  k=200   B12>=0.24 ratio>=1.45  F1=0.603
train  k=300   B12>=0.00 ratio>=1.30  F1=0.586
train  k=500   B12>=0.00 ratio>=1.30  F1=0.578
train  k=1000  B12>=0.00 ratio>=1.30  F1=0.529
train  k=2000  B12>=0.00 ratio>=1.25  F1=0.414
train  k=5000  B12>=0.00 ratio>=1.20  F1=0.242

FROZEN: {'level': 'patch', 'k': 200, 'b12_min': 0.24, 'ratio_min': 1.45, 'train_f1': 0.6034, 'tuned_on': 'train scenes only; F1 of fire class'}

VAL patch-level: n=504, fire patches=94 (0.187), 'flag all' F1=0.314
TP=56 FP=18 FN=38 TN=392 | precision=0.757 recall=0.596 F1=0.667
VAL pixel-level (B12>=0.10, ratio>=1.15): precision=0.508 recall=0.035 F1=0.066 IoU=0.034


In [8]:
import pandas as pd

hot_tr = tail[:, a_b, r_b]
det_tr = hot_tr >= k_b
fire_idx = np.where(y)[0]
sizes = ptr[fire_idx]
det = det_tr[fire_idx]

edges = [1, 100, 1000, 5000, 20000, 10**9]
rows = []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (sizes >= lo) & (sizes < hi)
    rows.append(dict(label_px=f"{lo}-{hi if hi < 10**9 else 'max'}", patches=int(m.sum()),
                     detected=int(det[m].sum()), recall=round(float(det[m].mean()), 3) if m.sum() else np.nan,
                     median_hot_px=int(np.median(hot_tr[fire_idx][m])) if m.sum() else 0))
print(pd.DataFrame(rows).to_string(index=False))

fp_idx = np.where(~y & det_tr)[0]
print(f"\nTrain false-alarm patches: {len(fp_idx)} of {int((~y).sum())}; "
      f"median hot px in them: {int(np.median(hot_tr[fp_idx])) if len(fp_idx) else 0}")

tp_t, fp_t = int((det_tr & y).sum()), int((det_tr & ~y).sum())
fn_t = int((~det_tr & y).sum())
res = pd.DataFrame([
    dict(split="train", level="patch", precision=tp_t/(tp_t+fp_t), recall=tp_t/(tp_t+fn_t), f1=f1_best),
    dict(split="val", level="patch", precision=pv, recall=rv, f1=2*pv*rv/(pv+rv)),
    dict(split="val", level="pixel", precision=0.508, recall=0.035, f1=0.066),
]).round(3)
res.to_csv(PROJECT_DIR / "results" / "baseline_threshold_results.csv", index=False)
print("\n", res.to_string(index=False))

  label_px  patches  detected  recall  median_hot_px
     1-100        0         0     NaN              0
  100-1000        1         0   0.000              0
 1000-5000       19         0   0.000              0
5000-20000       33        12   0.364              6
 20000-max      100        58   0.580            596

Train false-alarm patches: 9 of 1305; median hot px in them: 320

 split level  precision  recall    f1
train patch      0.886   0.458 0.603
  val patch      0.757   0.596 0.667
  val pixel      0.508   0.035 0.066
